# GGA AutoPILOT: Gemma 4 31B with macros and live controllers
Inputs: the competition data and gemma-4-31b-it-qat-w4a16-ct. Accelerator: RTX Pro 6000. Internet: off. Built by `make_notebook.py`, do not edit here.

In [ ]:
# Setup: installs vLLM and Chromium offline and starts Gemma 4 31B (4-bit) in a vLLM server on localhost:8000.
import io, json, os, subprocess, sys, threading, time, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

began = time.time()
root = Path('/kaggle/input')
harness = next(root.rglob('gga/protocol.py')).parents[1]
weights = next(root.rglob('gemma-4-31b-it-qat-w4a16-ct/*/model.safetensors')).parent
kit = next(root.rglob('kit/wheels')).parent
wheels = next(root.rglob('vllm-0.34.0-*.whl')).parent   # output of the notebook gga-vllm-wheels


def log(text):
    print(f'{time.time() - began:5.0f} s  {text}', flush=True)


# Chromium installs meanwhile. The input folder drops executable bits, so it is copied out first.
chromium = subprocess.Popen(f'cp -r {kit}/browsers /tmp && chmod -R +x /tmp/browsers'
                            f' && dpkg -i {kit}/debs/*.deb > /dev/null 2>&1', shell=True)
# The input folder is slow per stream but scales: read the weights in 16 streams, so vLLM loads from memory.
fd = os.open(weights / 'model.safetensors', os.O_RDONLY)
reading = ThreadPoolExecutor(16).map(lambda offset: os.pread(fd, 64 * 2**20, offset),
                                     range(0, os.fstat(fd).st_size, 64 * 2**20))
subprocess.run(f'pip install -q --no-index --find-links {wheels} --find-links {kit}/wheels vllm playwright',
               shell=True, check=True)
# vLLM brings torch for CUDA 13; the image's torchaudio (CUDA 12.8) then fails on import. Gemma needs no audio.
subprocess.run('pip uninstall -q -y torchaudio', shell=True, check=True)
log('installed vLLM and Playwright')
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1920x1080x24'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(harness))
sys.path.insert(0, '/kaggle/working')   # for autopilot.py, written by the next cell
os.environ.update(GGA_ROOT=str(harness), GGA_SEED='0', DISPLAY=':99', PLAYWRIGHT_BROWSERS_PATH='/tmp/browsers')
# FlashInfer's sampler compiles at startup and fails to link (-lcuda missing); greedy needs no sampler anyway
os.environ['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
list(reading)   # raises if a read failed
os.close(fd)
log('read the weights')

server = subprocess.Popen(
    ['vllm', 'serve', str(weights), '--port', '8000', '--served-model-name', 'gemma', '--max-model-len', '16384',
     '--gpu-memory-utilization', '0.9', '--max-num-seqs', '8', '--limit-mm-per-prompt', '{"image": 6}'],
    stdout=open('/kaggle/working/vllm.log', 'w'), stderr=subprocess.STDOUT)
while True:   # the competition allows 15 minutes for everything before the inference server listens
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2)
        break
    except OSError:
        if server.poll() is not None:
            sys.exit('vLLM exited:\n' + Path('/kaggle/working/vllm.log').read_text()[-3000:])
        time.sleep(2)
log('vLLM is serving Gemma')
if chromium.wait():
    raise RuntimeError('Installing Chromium failed')
log('ready')


In [ ]:
%%writefile /kaggle/working/autopilot.py
"""AutoPILOT-style agent for the Generalist Gaming Agent competition.

step() never waits for the model. The model (`think`) runs in a background thread and is only woken by
events. Its decision is one game action, a controller (Python code it writes, which then plays every
decision point from the screen), a control check, KEEP or REUSE of a stored controller.

Without the model, about once a second, the agent measures image signals: new screen content, changes in
the progress display the model located, repeated actions, screens seen before (game over), and wakes the
model when nothing moves for STUCK_S. Every controller is kept per game in a library with its results;
after a game over the agent replays the actions that left that screen and restarts the best controller of
the next screen at once, and it tries model-written variants of a working controller in parallel.
"""
import io
import json
import time
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from PIL import Image
from gga.protocol import Action

CHECK_S = 10               # while a controller plays, the model looks at it this often
CONTROLLER_MAX_S = 0.1     # a controller call slower than this is switched off
SCENE_DIFF = 0.5           # share of pixels that must differ from ~1 s ago for a scene change
SCENE_COOLDOWN_S = 2
SCENE_MATCH = 12           # mean abs difference of 16x16 gray thumbnails under which two screens are one scene
NOVEL_DIFF = 10            # ... over which a screen counts as new content
DISPLAY_DIFF = 6           # mean abs change in the progress display box that counts as the display changing
STUCK_S = 15               # no progress signal this long: wake the model
PROBE_WAIT = 4             # decision points (200 ms) to watch after each control-check action
REPLAY_GAP_S = 0.6         # between replayed restart actions
VARY_AFTER_S = 20          # a controller running this long gets a variant tried
TRIAL_S = 15               # how long a variant plays before it is compared with its parent
BEST_MIN_S = 5             # a controller must have played this long to be restarted as the best
XY = {'x', 'y', 'x2', 'y2'}
GRID = 1000                # the model's coordinates: 0-999 over the screen, in actions and controllers alike


def to_pixels(params, width, height):
    """0-999 grid coordinates -> the game's pixels, for an action's params and its sequence steps."""
    for target in [params, *params.get('steps', [])]:
        for key, size in (('x', width), ('y', height), ('x2', width), ('y2', height)):
            if target.get(key) is not None:
                target[key] = min(max(int(target[key]), 0), GRID - 1) * (size - 1) // (GRID - 1)
    return params


def thumb(image):
    return np.asarray(image.convert('L').resize((16, 16)), dtype=np.float32).ravel()


class Agent:
    def __init__(self):
        self.pool = ThreadPoolExecutor(1)
        self.variant_pool = ThreadPoolExecutor(1)   # its calls run beside the model's; vLLM batches them
        self.last_t = float('inf')
        self._reset()

    def think(self, obs, event, memory):
        """Override with the model. Returns one of
        {"tool": ..., "params": {...}}         an action, in pixels
        {"controller": "<code>"}              code defining control(frame, mem) -> action dict or None
        {"probe": [(label, action or None)]}  a control check: each action once, then watch what changes
        {"keep": True}                        the running controller plays on
        {"reuse": <library id>}               restart a stored controller

        frame is the screen scaled to GRID x GRID, RGB numpy, so frame[y, x] is grid point (x, y);
        mem is a dict it keeps. Its actions use grid coordinates, which the agent turns into pixels.
        To say why it hands back, it sets mem['handoff'] = '<reason>' before returning None.
        """
        return {'tool': 'skip', 'params': {}}

    def vary(self, obs, entry, memory):
        """Override with the model: code of a variant of library entry `entry` that should do better, or None."""
        return None

    def _reset(self):
        self.future = self.variant = None   # shortcut: a call still running from the last game delays this one
        self.memory = {}                    # per game, shared with the model
        self.library = self.memory['library'] = []
        self.last_frames = self.memory['last_frames'] = deque(maxlen=8)   # (game time, jpeg), one a second
        self.attempts = self.memory['attempts'] = []                      # durations of ended attempts
        self.controller = self.entry = self.trial = None
        self.probes, self.queue, self.queue_t = [], [], 0
        self.wake = 'new game'
        self.events = []                    # (game time, event) log, for debugging
        self.scenes = deque()
        self.scene_t = -SCENE_COOLDOWN_S
        self.scene_thumbs, self.scene_id, self.scene_actions, self.exits = [], None, [], {}
        self.signal_t, self.seen_thumbs, self.box = -1, np.zeros((0, 256), np.float32), None
        self.novel_t = self.display_t = self.stuck_t = self.attempt_t = 0.0
        self.recent = deque(maxlen=10)      # the model's last actions, as (tool, key) patterns
        self.restart_t, self.restart_delays = None, []

    def step(self, obs, tools):
        try:
            return self._step(obs)
        except Exception as e:   # never raise: the harness plays skip and may start a fresh agent without memory
            self.events.append((getattr(obs, 'elapsed_s', None), f'agent error: {e!r}'))
            return Action('skip')

    def _step(self, obs):
        t = obs.elapsed_s
        if t < self.last_t:   # the clock started again: a new game
            self._reset()
        self.last_t = t
        image = Image.open(io.BytesIO(obs.image)).convert('RGB')
        self._check_scene(t, np.asarray(image), image)
        self._signals(t, image, obs.image)
        if self.controller and obs.last_action.get('ok') is False:   # the game refused its last action
            error = obs.last_action.get('error')
            self._stop(t, 'refused', error, wake=f"the game refused the controller's action: {error}")

        action = None
        if self.future and self.future.done():
            future, self.future = self.future, None
            try:
                action = self._apply(future.result(), t, image)
            except Exception as e:   # bad model reply: tell the model on its next call
                self._wake(t, f'your last reply failed: {e!r}')
        if self.variant and self.variant.done():
            future, self.variant = self.variant, None
            try:
                code = future.result()
                if code and action is None and self.entry and not self.trial:
                    action = self._start_trial(code, t, image)
            except Exception as e:
                self.events.append((t, f'variant failed: {e!r}'))
        if action is None and self.trial and t >= self.trial['end']:
            action = self._end_trial(t, image)
        if action is None and self.probes:
            action = self._probe(t, image)
        elif action is None and self.queue and t >= self.queue_t:
            action, self.queue_t = self.queue.pop(0), t + REPLAY_GAP_S
        if action is None and self.controller:
            action = self._run_controller(t, image)
        busy = self.controller or self.probes or self.queue or obs.program_left_s
        if not busy and self.future is None and not self.wake:
            self._wake(t, 'ready')   # nothing is playing: ask for the next move
        if self.wake and self.future is None:
            self.future = self.pool.submit(self.think, obs, self.wake, self.memory)
            self.future_scene, self.wake = self.scene_id, None
        e = self.entry
        if (e and not self.trial and self.variant is None and self.runtime(e, t) >= VARY_AFTER_S
                and t >= e.get('vary_t', 0)):
            e['vary_t'] = t + 30
            self.variant = self.variant_pool.submit(self.vary, obs, e, self.memory)
        return action or Action('skip')   # skip lets running inputs play on

    def _wake(self, t, event):
        self.events.append((t, event))
        self.wake = self.wake or event

    # --- decisions ---

    def _apply(self, decision, t, image):
        if decision.get('keep'):
            return None
        if 'probe' in decision:
            self._stop(t, 'ok', 'replaced by a control check')
            self.probes = [{'label': label, 'action': action and Action(action['tool'], action.get('params') or {})}
                           for label, action in decision['probe']]
            self.queue = []
            self.memory['probes'] = []
            return self._probe(t, image)
        if 'reuse' in decision:
            entry = next((e for e in self.library if e['id'] == decision['reuse'] and 'fn' in e), None)
            if entry is None:
                raise ValueError(f"there is no working controller #{decision['reuse']}")
            self._start(entry, t)
            return self._run_controller(t, image)
        if 'controller' in decision:
            self._start(self._new_entry(decision['controller'], t), t)
            return self._run_controller(t, image)
        self._stop(t, 'ok', 'replaced by an action')
        self.probes, self.queue = [], []
        action = Action(decision['tool'], decision.get('params') or {})
        self.recent.append((action.tool, str(action.params.get('key') or action.params.get('keys') or '')))
        if self.future_scene == self.scene_id:   # decided on an earlier screen: it did not lead off this one
            self.scene_actions.append(action)
        return action

    # --- controllers and their library ---

    def _new_entry(self, code, t, parent=None):
        first = code.strip().splitlines()[0] if code.strip() else ''
        entry = {'id': len(self.library) + 1, 'scene': self.scene_id, 'code': code, 'parent': parent,
                 'desc': first.lstrip('#').replace('skill:', '').strip() if first.startswith('#') else first[:60],
                 'runtime': 0.0, 'events': 0, 'status': 'new', 'reason': ''}
        self.library.append(entry)
        namespace = {'np': np}
        try:
            exec(code, namespace)   # trusted: only our own model writes this code
            entry['fn'] = namespace['control']
        except Exception as e:
            entry['status'], entry['reason'] = 'crashed', repr(e)
            raise
        return entry

    def runtime(self, entry, t):
        return entry['runtime'] + (t - entry['since'] if entry['status'] == 'running' else 0)

    def rate(self, entry, t):
        """Progress signals (new screen content, display changes) per minute while it played."""
        return entry['events'] * 60 / max(self.runtime(entry, t), 1)

    def _best(self, scene, t):
        good = [e for e in self.library if e['scene'] == scene and e['status'] == 'ok' and e['events']
                and self.runtime(e, t) >= BEST_MIN_S]
        return max(good, key=lambda e: self.rate(e, t), default=None)

    def _start(self, entry, t):
        if self.entry is not entry:
            self._stop(t, 'ok', f"replaced by #{entry['id']}")
        self.controller, self.entry, self.controller_mem = entry['fn'], entry, {}
        self.controller_start = self.controller_t = entry['since'] = t
        entry['status'], self.probes, self.queue = 'running', [], []

    def _stop(self, t, status, reason, wake=None):
        e = self.entry
        if e:
            e['runtime'] = self.runtime(e, t)
            e['status'], e['reason'] = status, reason
            if status == 'ok' and e.get('stuck'):
                e['status'] = 'no progress'
        self.controller = self.entry = None
        if wake:
            self._wake(t, wake)

    def _run_controller(self, t, image):
        frame = np.asarray(image.resize((GRID, GRID), Image.NEAREST))   # NEAREST: no blended colors at edges
        start = time.time()
        try:
            # shortcut: an endless loop in control() hangs step(); move it to a subprocess if models write those
            choice = self.controller(frame, self.controller_mem)
            if choice is not None:   # numpy -> plain numbers, coordinates -> ints, no empty keys: as the game requires
                choice = json.loads(json.dumps(choice, default=lambda o: o.item()), object_hook=lambda d: {
                    k: round(v) if k in XY and isinstance(v, float) else [x for x in v if x != ''] if k == 'keys'
                    and isinstance(v, list) else v for k, v in d.items()})
                choice['params'] = to_pixels(choice.get('params') or {}, *image.size)
        except Exception as e:
            self._stop(t, 'crashed', repr(e), wake=f'controller crashed: {e!r}')
            return None
        took = time.time() - start
        if took > CONTROLLER_MAX_S:
            self._stop(t, 'too slow', f'{took:.2f} s per call', wake=f'controller too slow ({took:.2f} s per call), stopped')
            return None
        if choice is None:
            reason = self.controller_mem.get('handoff', 'controller handed control back')
            self._stop(t, 'ok', reason, wake=reason)
            return None
        if self.restart_t is not None:   # measures how fast play resumes after a game over
            self.restart_delays.append(round(t - self.restart_t, 1))
            self.restart_t = None
        if t - self.controller_t >= CHECK_S and not self.trial:
            self.controller_t = t   # it keeps playing while the model looks
            self._wake(t, f'check-in: your controller has played for {t - self.controller_start:.0f} s and plays on '
                          'while you look. Is it making progress on what counts (your game note)? '
                          'KEEP it, or reply with a better controller or an action')
        return Action(choice['tool'], choice.get('params') or {})

    def _start_trial(self, code, t, image):
        parent = self.entry
        entry = self._new_entry(code, t, parent=parent['id'])
        self._start(entry, t)
        parent['status'] = 'ok'   # paused, not failed
        self.trial = {'entry': entry, 'parent': parent, 'end': t + TRIAL_S}
        self.events.append((t, f"trying variant #{entry['id']} of #{parent['id']}"))
        return self._run_controller(t, image)

    def _end_trial(self, t, image):
        trial, self.trial = self.trial, None
        variant, parent = trial['entry'], trial['parent']
        if self.entry is not variant:   # it was stopped or replaced meanwhile
            return None
        if variant['events'] and self.rate(variant, t) > 1.2 * self.rate(parent, t):
            self.events.append((t, f"variant #{variant['id']} beats #{parent['id']}: kept"))
            return None
        self._stop(t, f"worse than #{parent['id']}", f'{self.rate(variant, t):.0f} vs {self.rate(parent, t):.0f} progress/min')
        self.events.append((t, f"variant #{variant['id']} is not better than #{parent['id']}: back to #{parent['id']}"))
        self._start(parent, t)
        return self._run_controller(t, image)

    # --- control check ---

    def _probe(self, t, image):
        small = np.asarray(image.convert('L').resize((100, 100), Image.NEAREST), np.float32)
        probe = self.probes[0]
        if 'before' not in probe:
            probe['before'], probe['n'] = small, 0
            return probe['action']   # None: watch the screen without input, for its own motion
        probe['n'] += 1
        if probe['n'] < PROBE_WAIT:
            return None
        changed = np.abs(small - probe['before']) > 30
        if changed.mean() < 0.005:
            result = 'nothing visible changed'
        else:
            ys, xs = np.nonzero(changed)
            result = (f'{changed.mean():.0%} of the screen changed, around x {xs.min() * 10}-{xs.max() * 10 + 9}, '
                      f'y {ys.min() * 10}-{ys.max() * 10 + 9}')
        self.memory['probes'].append(f"{probe['label']}: {result}")
        self.probes.pop(0)
        if not self.probes:
            self._wake(t, 'control check done: see its results')
        return None

    # --- image signals, about once a second ---

    def _signals(self, t, image, jpeg):
        if t - self.signal_t < 1:
            return
        self.signal_t = t
        self.last_frames.append((t, jpeg))
        progress = False
        small = thumb(image)
        if not len(self.seen_thumbs) or np.abs(self.seen_thumbs - small).mean(axis=1).min() > NOVEL_DIFF:
            self.novel_t, progress = t, True
            if len(self.seen_thumbs) < 3000:
                self.seen_thumbs = np.vstack([self.seen_thumbs, small])
        box = self.memory.get('display_box')   # the progress display the model located, grid x0, y0, x1, y1
        if box:
            px = to_pixels({'x': box[0], 'y': box[1], 'x2': box[2], 'y2': box[3]}, *image.size)
            crop = np.asarray(image.convert('L').crop((px['x'], px['y'], px['x2'] + 1, px['y2'] + 1))
                              .resize((32, 8)), np.float32)
            if self.box is not None and np.abs(crop - self.box).mean() > DISPLAY_DIFF:
                self.display_t, progress = t, True
            self.box = crop
        if progress and self.entry:
            self.entry['events'] += 1
        pattern, count = Counter(self.recent).most_common(1)[0] if self.recent else (None, 0)
        repeating = count >= 8
        self.memory['signals'] = (
            f'new screen content {t - self.novel_t:.0f} s ago; '
            + (f'progress display changed {t - self.display_t:.0f} s ago; ' if box else 'no progress display located; ')
            + (f'{count} of your last 10 actions were {pattern[0]} {pattern[1]}; ' if repeating else '')
            + f'attempt {len(self.attempts) + 1}, running {t - self.attempt_t:.0f} s'
            + (', earlier attempts lasted ' + ', '.join(f'{a:.0f} s' for a in self.attempts[-5:]) if self.attempts else ''))
        quiet = t - self.novel_t >= STUCK_S and (repeating or not box or t - self.display_t >= STUCK_S)
        if quiet and t - self.stuck_t >= STUCK_S and not self.probes:
            self.stuck_t = t
            if self.entry:
                self.entry['stuck'] = self.entry.get('stuck', 0) + 1
            self._wake(t, f"no progress signal: {self.memory['signals']}. Try something fundamentally different, "
                          'e.g. controls from your note you have not used yet')

    # --- scenes: screen changes, game over, restart ---

    def _scene_of(self, small):
        if self.scene_thumbs:
            dist = np.abs(np.array(self.scene_thumbs) - small).mean(axis=1)
            if dist.min() < SCENE_MATCH:
                return int(dist.argmin()), True
        self.scene_thumbs.append(small)
        return len(self.scene_thumbs) - 1, False

    def _check_scene(self, t, frame, image):
        if self.scene_id is None:
            self.scene_id = self._scene_of(thumb(image))[0]
        small = frame[::8, ::8].mean(axis=2)
        self.scenes.append((t, small))
        while self.scenes[0][0] < t - 1:
            old = self.scenes.popleft()[1]
            if t - self.scene_t < SCENE_COOLDOWN_S or len(self.scenes) < 2:
                continue
            changed = (np.abs(small - old) > 24).mean()
            holds = (np.abs(small - self.scenes[-2][1]) > 24).mean() < 0.05
            if changed > SCENE_DIFF and holds:
                self.scene_t = t
                self._new_scene(t, image)

    def _new_scene(self, t, image):
        old = self.scene_id
        new, known = self._scene_of(thumb(image))
        if self.scene_actions:            # what the model did on the old screen led off it: keep it for replays
            self.exits[old] = self.scene_actions[-4:]
            self.scene_actions = []
        if old in self.exits:             # left a menu / game-over screen: a new attempt starts
            self.attempt_t = t
        self._stop(t, 'ok', 'the scene changed')
        self.scene_id = new
        if new in self.exits and new != old:   # back on a screen left before: game over (or menu)
            if self.attempt_t:
                self.attempts.append(t - self.attempt_t)
            self.restart_t = t
            self.queue, self.queue_t = list(self.exits[new]), t + REPLAY_GAP_S
            self._wake(t, f'game over: back on screen #{new} after {self.attempts[-1] if self.attempts else t:.0f} s; '
                          f'replaying the {len(self.queue)} actions that left it last time. '
                          'What killed you or ended the run? What do you change?')
            return
        best = self._best(new, t)
        if best:
            self._start(best, t)
        self._wake(t, f"the scene changed (screen #{new}, {'seen before' if known else 'new'})"
                      + (f"; your best controller for it, #{best['id']}, is playing again" if best else ''))


In [ ]:
# Notebook cell: Gemma, served by vLLM on localhost (setup cell), as the model behind autopilot.Agent.
import base64
import io
import json
import re
import time
import urllib.request

from PIL import Image

from autopilot import Agent as AutoPilot, to_pixels

SERVER = 'http://localhost:8000/v1/chat/completions'

TOOLS = '''Tools. Coordinates are on a 0-999 grid over the screen (x: left to right, y: top to bottom).
Durations are in ms, at most 10000. Keys: a printable character or a name such as ArrowLeft, Space, Enter,
Shift, Escape. Every reply except skip replaces what is still playing.
{"tool":"click","params":{"x":500,"y":500}}                     click; optional "button": "left", "middle" or "right"
{"tool":"move","params":{"x":500,"y":500}}                      move the pointer without clicking (aiming, hovering)
{"tool":"drag","params":{"x":100,"y":500,"x2":800,"y2":500,"hold_ms":100}}   press, move to x2,y2, release
{"tool":"scroll","params":{"x":500,"y":500,"dy":100}}           mouse wheel; dy > 0 down, 100 = one notch; optional dx
{"tool":"press_key","params":{"key":"Space","hold_ms":100}}     tap a key; hold it longer to keep moving
{"tool":"hold_keys","params":{"keys":["ArrowRight","Space"],"duration_ms":500}}   hold keys together, then release
{"tool":"key_down","params":{"key":"ArrowRight"}}               hold a key until key_up
{"tool":"key_up","params":{"key":"ArrowRight"}}                 release a key held by key_down
{"tool":"wait","params":{"duration_ms":500}}                    stop what is playing and idle
{"tool":"skip","params":{}}                                     change nothing: what is playing plays on
{"tool":"sequence","params":{"steps":[{"keys":["ArrowRight"],"buttons":[],"x":500,"y":500,"duration_ms":400},{"keys":["ArrowUp","x"],"buttons":["left"],"x":500,"y":500,"duration_ms":200}]}}
    a timed program of up to 200 steps that plays while you think. Each step holds exactly its keys and
    buttons (all else is released) for duration_ms; x, y are optional. The last step stays held.'''

RULES = f'''You play a real-time browser game from screenshots. The game keeps running while you think, and each
of your replies takes seconds, so prefer moves that keep playing on their own.

{TOOLS}

Start every reply with one line: Progress: <what counts in this game (your game note), as it stands on this
screen, e.g. "score 120", "alive, 3 lives", "level 2", "distance 450", "timer 1:20", or "unknown">
If a line of your game note is wrong, add a line: Note: <the corrected line, e.g. "Progress display: 0,930,999,999">
After a game over, add a line: Lesson: <what ended the run and what you change>
Then reply in exactly one of these four forms.

1. One action, as JSON on one line, from the tools above.

2. A controller: Python code that plays every 200 ms from the screen without you. Use it when the game
   needs fast reactions (dodging, aiming, steering, timing jumps).
```python
# skill: <one line: what this controller does>
def control(frame, mem):
    # frame: numpy uint8 array (1000, 1000, 3), RGB: the screen scaled to the 0-999 grid, so frame[y, x]
    #        is grid point (x, y). np.argwhere returns (y, x). np is numpy.
    # mem: a dict kept between calls.
    # Return one action dict from the tools above, in grid coordinates as they come from frame,
    # or set mem["handoff"] = "<why>" and return None to give control back to you.
    ...
```
   Rules for controllers:
   - The first line is "# skill: <one line>"; it is the only comment. No other comments, docstrings or
     blank lines: every token you write costs game time.
   - Use the coordinates from frame as they are: frame already is the grid, never rescale them.
   - Act on what you detect: aim at the targets you found, move away from the threats you found.
     Code that finds things and then ignores them is useless.
   - Serve what counts in this game (your game note): survive, score, go fast or get far.
   - Fast, well under 0.1 s per call: numpy color masks, never Python loops over pixels.
   - Give control back when something happens it cannot handle.

3. KEEP
   Only while your controller is playing and it works: it plays on unchanged. When what counts is
   survival and you are still alive, that is progress.

4. REUSE #<id>
   Restart a controller from your library below.

Prefer a sequence when it is enough. Test an uncertain control briefly with one action before you rely on it.
If a menu or game-over screen shows, start the game the way the brief or the screen says.
If something you tried did not help, try something different. Never repeat what your library lists as failed.'''

NOTE = '''Look at the screen and the brief, then write a short game note, exactly these six lines:
Type: <the kind of game, e.g. shooter, platformer, puzzle, racing, strategy, idle, card>
Goal: <what you must do to win or do well>
Controls: <the keys and mouse actions that matter, and what each does>
Progress display: <x0,y0,x1,y1: the box on the 0-999 grid around the score, lives, level, money, timer or
  bar, usually at a screen edge; or none if this screen shows none>
What counts: <what to maximize: survival, points, time, distance, levels, money, or what else>
Strategy: <from what counts: for survival take few risks; for points try a lot; and your first plan>'''

REVISE = NOTE + '''
The game is running now. Your note so far is below; correct every line this screen shows better.'''

PROBE = '''The game is running. Plan a quick control check: one test action for each control the brief or your
note names (each key, each mouse action), so you see what each one does. Skip any that restart, pause,
quit, or open settings or menus. At most 8 lines, each: <control>: <one JSON action from the tools>
Example:
Q (build): {"tool":"press_key","params":{"key":"q"}}
left click (dig): {"tool":"click","params":{"x":500,"y":500}}'''

VARY = '''Below is one of your controllers that works. Write a variant that should do clearly better at what
counts, e.g. the same plus dodging, better aim, or using a control it ignores. Reply with one python
block only, first line "# skill: <one line>", same rules as before.'''


def parse(reply, width, height):
    """Gemma's reply -> a decision for autopilot.Agent (actions in pixels). Raises ValueError if none."""
    code = re.search(r'```python\n(.*?)(```|$)', reply, re.S)
    if code:
        return {'controller': code.group(1)}
    start = reply.find('{')
    if start < 0:
        reuse = re.search(r'\bREUSE\s*#?(\d+)', reply)
        if reuse:
            return {'reuse': int(reuse.group(1))}
        if re.search(r'\bKEEP\b', reply):
            return {'keep': True}
        raise ValueError('reply had no JSON action, python block, KEEP or REUSE')
    choice = json.JSONDecoder().raw_decode(reply[start:])[0]
    return {'tool': choice['tool'], 'params': to_pixels(choice.get('params') or {}, width, height)}


def parse_probe(reply, width, height):
    """'<label>: {json}' lines -> [(label, action in pixels)], watching the screen alone first."""
    probes = [('no input (the screen alone)', None)]
    for line in reply.splitlines():
        label, _, rest = line.partition(':')
        start = rest.find('{')
        if start < 0:
            continue
        try:
            action = json.JSONDecoder().raw_decode(rest[start:])[0]
            probes.append((label.strip()[:40], {'tool': action['tool'],
                                                'params': to_pixels(action.get('params') or {}, width, height)}))
        except (ValueError, KeyError, TypeError, AttributeError):
            continue
    return probes[:9]


def display_box(text):
    """The 'Progress display: x0,y0,x1,y1' box of a note, or None."""
    found = re.findall(r'Progress display:\s*\(?(\d+)\D+(\d+)\D+(\d+)\D+(\d+)', text)
    if not found:
        return None
    x0, y0, x1, y1 = (min(int(v), 999) for v in found[-1])   # the last one: a correction overrides the note
    return (x0, y0, x1, y1) if x0 < x1 and y0 < y1 else None


def chat(messages, max_tokens):
    """One reply from the vLLM server: (text, tokens generated)."""
    body = json.dumps({'model': 'gemma', 'messages': messages, 'max_tokens': max_tokens, 'temperature': 0}).encode()
    request = urllib.request.Request(SERVER, body, {'Content-Type': 'application/json'})
    with urllib.request.urlopen(request, timeout=85) as response:
        out = json.load(response)
    return out['choices'][0]['message']['content'], out['usage']['completion_tokens']


class Agent(AutoPilot):
    replies = []   # every reply of every game, for reading after a run

    def ask(self, obs, event, system, text, max_tokens, images=()):
        content = []
        for jpeg in [*images, obs.image]:
            frame = Image.open(io.BytesIO(jpeg)).convert('RGB')
            frame.thumbnail((768, 768))
            out = io.BytesIO()
            frame.save(out, 'JPEG', quality=90)
            content.append({'type': 'image_url', 'image_url': {'url': 'data:image/jpeg;base64,'
                                                                      + base64.b64encode(out.getvalue()).decode()}})
        content.append({'type': 'text', 'text': text})
        began = time.time()
        reply, tokens = chat([{'role': 'system', 'content': system}, {'role': 'user', 'content': content}], max_tokens)
        self.replies.append((obs.elapsed_s, event, reply, time.time() - began, tokens))
        return reply

    def library_text(self, memory, t):
        lines = []
        for e in memory['library']:
            if e['status'] in ('ok', 'running') or e['status'].startswith('worse'):
                lines.append(f"#{e['id']} [screen {e['scene']}] {e['desc']}: played {self.runtime(e, t):.0f} s, "
                             f"{self.rate(e, t):.0f} progress signals/min, {e['status']}")
            else:
                lines.append(f"#{e['id']} {e['desc']}: FAILED ({e['status']}: {e['reason'][:100]})")
        return '\n'.join(lines[-12:]) or 'empty'

    def note_text(self, memory):
        return memory['note'] + ''.join(f'\nCorrection: {fix}' for fix in memory.get('fixes', [])[-3:])

    def think(self, obs, event, memory):
        info, t = obs.info, obs.elapsed_s
        brief = f"Brief: {info['description']} Controls: {info['controls']}\n"
        if 'note' not in memory:   # at the very start, often a menu: rewritten once the game shows
            memory['note'] = self.ask(obs, 'game note', NOTE, brief, 250).strip()
        elif ((event.startswith('the scene changed') and not memory.get('note_after_scene'))
              or (t >= 20 and not memory.get('note_at_20s'))):
            memory['note_after_scene' if event.startswith('the scene changed') else 'note_at_20s'] = True
            memory['note'] = self.ask(obs, 'game note revised', REVISE,
                                      brief + 'Your note so far:\n' + self.note_text(memory), 250).strip()
            memory['fixes'] = []
        memory['display_box'] = display_box(self.note_text(memory))
        if not memory.get('probed') and (memory.get('note_after_scene') or memory.get('note_at_20s')):
            memory['probed'] = True   # once per game, as soon as the game itself runs
            probes = parse_probe(self.ask(obs, 'control check', PROBE + '\n\n' + TOOLS,
                                          brief + 'Your game note:\n' + self.note_text(memory), 300),
                                 info['width'], info['height'])
            if len(probes) > 1:
                return {'probe': probes}

        log = memory.setdefault('log', [])
        progress = memory.setdefault('progress', [])
        lessons = memory.setdefault('lessons', [])
        game_over = event.startswith('game over')
        text = (
            brief + f"Your game note:\n{self.note_text(memory)}\n"
            + ('Control check, what each control changed on screen:\n' + '\n'.join(memory['probes']) + '\n'
               if memory.get('probes') else '')
            + f"Time left: {info['remaining_s']:.0f} s.\n"
            f"What just happened: {event}.\n"
            f"Result of the last action: {obs.last_action}\n"
            f"Measured on screen: {memory.get('signals', 'nothing yet')}\n"
            + ('Lessons from earlier runs: ' + ' | '.join(lessons[-3:]) + '\n' if lessons else '')
            + "Progress you read so far (game time: progress): "
            + (', '.join(f'{pt:.0f} s: {p}' for pt, p in progress[-10:]) or 'none') + '\n'
            + f"Your controller library:\n{self.library_text(memory, t)}\n"
            + (f"Playing now: #{self.entry['id']}, for {t - self.controller_start:.0f} s. Its code:\n"
               f"```python\n{self.entry['code']}```\n" if self.controller and self.entry else '')
            + ('The first images show the last seconds before the game over, oldest first; the last image is now.\n'
               if game_over else '')
            + "Your recent replies, oldest first:\n" + ('\n'.join(log[-6:]) or 'none') + '\nYour reply:'
        )
        before = [jpeg for _, jpeg in list(memory['last_frames'])[-6:-1]] if game_over else []
        reply = self.ask(obs, event, RULES, text, 768, before)
        log.append(f'[{event[:60]}] ' + ' '.join(reply.split())[:300])
        for key, store in (('Progress', progress), ('Lesson', lessons), ('Note', memory.setdefault('fixes', []))):
            found = re.search(rf'^\s*{key}:\s*(.+)$', reply, re.M)
            if found:
                store.append((t, found.group(1).strip()[:40]) if key == 'Progress' else found.group(1).strip()[:200])
        return parse(reply, info['width'], info['height'])

    def vary(self, obs, entry, memory):
        info = obs.info
        text = (f"Brief: {info['description']} Controls: {info['controls']}\n"
                f"Your game note:\n{self.note_text(memory)}\n"
                f"Controller #{entry['id']} ({entry['desc']}), {self.rate(entry, obs.elapsed_s):.0f} progress signals/min:\n"
                f"```python\n{entry['code']}```\n{VARY}")
        code = re.search(r'```python\n(.*?)(```|$)', self.ask(obs, f"variant of #{entry['id']}", RULES, text, 768), re.S)
        return code and code.group(1)


In [ ]:
# Plays the sample games here, one Agent for all as in the competition, and prints a measurement row per game.
import statistics
from collections import Counter
from IPython.display import HTML, Image as Frame, display
from gga.game import Game

GAMES = ['asteroid_capitalist', 'blarglike', 'cavebots', 'off_line']
PLAY_SECONDS = 120             # per game; in the competition a game runs for 10 minutes

# The competition's run skips this: the server in the last cell must be up within 15 minutes.
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    specs = json.load(open(harness / 'games.json'))['games']
    agent = Agent()
    Agent.replies.clear()
    screen = display(HTML('<i>Starting...</i>'), display_id=True)
    status = display(HTML(''), display_id=True)
    sheet = []
    log = open('/kaggle/working/replies.txt', 'w')   # every Gemma reply, to read after the run
    for name in GAMES:
        game = Game(name, video_dir=f'/kaggle/working/video_{name}')
        game.initialize({'minutes': PLAY_SECONDS / 60})
        tools = game.tools()
        first_reply = len(Agent.replies)
        turns, refused, step_s, tools_used = 0, [], [], Counter()
        while not game.is_terminal():
            obs = game.get_observation()
            began = time.time()
            action = agent.step(obs, tools)
            step_s.append(time.time() - began)
            result = game.handle_tool(action.tool, action.params)
            turns += 1
            tools_used[('controller: ' if agent.controller else '') + action.tool] += 1
            if not result['ok'] and not game.is_terminal():
                refused.append(result.get('error'))
            if turns % 5 == 0:
                screen.update(Frame(data=obs.image, width=640))
                status.update(HTML(f'{name}: {obs.elapsed_s:.0f} s, controller {"on" if agent.controller else "off"}, '
                                   f'last event: <code>{agent.events[-1][1][:120] if agent.events else ""}</code>'))
        game.stop()
        score = game.get_score()   # local runs only: the game's own counter, normalized 0..1 like the leaderboard
        replies = Agent.replies[first_reply:]
        writes = [r for r in replies if '```python' in r[2]]
        stuck = [(round(t), e[:60]) for t, e in agent.events if e.startswith('no progress')]
        row = {
            'game': name, 'metric': specs[name]['scoring']['metric'], 'score': score,
            'skip share': round(tools_used['skip'] / max(turns, 1), 2), 'refused': len(refused), 'refusals': refused[:3],
            'step max ms': round(max(step_s) * 1000),
            'replies': len(replies), 'reply s median': round(statistics.median(r[3] for r in replies), 1) if replies else None,
            'tokens median': statistics.median(r[4] for r in replies) if replies else None,
            'controller writes/min': round(len(writes) / (PLAY_SECONDS / 60), 1),
            'attempts': len(agent.attempts) + 1, 'attempt s': [round(a) for a in agent.attempts],
            'restart → controller s': agent.restart_delays, 'no-progress wakes': stuck,
            'library': [(e['id'], e['desc'][:40], e['status'], round(agent.rate(e, obs.elapsed_s)))
                        for e in agent.library],
            'probes': agent.memory.get('probes'), 'note': agent.memory.get('note'),
            'lessons': agent.memory.get('lessons'), 'actions': dict(tools_used),
        }
        sheet.append(row)
        print(json.dumps(row, indent=1, ensure_ascii=False))
        print('Events:', [(round(t, 1), e[:100]) for t, e in agent.events])
        log.write(f'##### {name}\n')
        for elapsed, event, reply, secs, tokens in replies:
            log.write(f'=== {elapsed:.1f} s, {event} ({secs:.1f} s, {tokens} tokens)\n{reply}\n\n')
    log.close()
    json.dump(sheet, open('/kaggle/working/sheet.json', 'w'), indent=1, ensure_ascii=False, default=str)
    print('\nMeasurement sheet')
    for row in sheet:
        print(f"{row['game']:20} score {row['score']}  skip {row['skip share']}  refused {row['refused']}  "
              f"reply {row['reply s median']} s / {row['tokens median']} tok  writes/min {row['controller writes/min']}  "
              f"attempts {row['attempts']}  no-progress {len(row['no-progress wakes'])}  restart {row['restart → controller s']}")


In [ ]:
# Hands the agent to the competition. Save & Run All, then submit that version.
from kaggle_evaluation.gga.gga_inference_server import GGAInferenceServer

Path('/kaggle/working/agent.py').write_text('from __main__ import Agent\n')  # the server loads the agent from this file
GGAInferenceServer().run()
